# exp14 — Mechanistic Validation of the Paragraph-Start Acrostic Mechanism

**Goal:** close the main mechanistic gaps after `exp13d` using the same successful triplet dataset from `exp11`.

This notebook runs five causal checks:

1. **Source masking** — mask attention to previous paragraph starts, payload-letter tokens, or the full scheme span.
2. **Previous token vs long-range** — separate local context (last content token before paragraph break) from long-range structure (previous paragraph-start positions).
3. **Scheme granularity** — compare `letters only` vs `scheme without letters` vs `full scheme`.
4. **Head specialization** — ablate top heads and compare with random heads.
5. **Attention rewrite** — redirect source-set attention mass to non-source prefix tokens.

**Primary metric:** `P(letter)` at the paragraph-start prediction position.

For each paragraph start, we take the final logits at the token **just before** the paragraph start, apply softmax, and sum the probability mass over all vocabulary tokens whose decoded form starts with the required letter.

For each intervention we report:

- `baseline_mean`
- `changed_mean`
- `delta = baseline_mean - changed_mean`
- paired `t` and `p`
- paired effect size `Cohen dz = mean(delta_i) / std(delta_i)`

**Interpretation:**

- large positive `delta` means the removed or rewritten source was causally helping the model produce the required start letter
- `stego ≈ control` means the mechanism is about **constraint-following**, not hiddenness

**Data:** `results/exp11/exp11_pairs_with_control.json`

**Recommended runtime:** A100. Start with `MAX_PAIRS = 20` for a smoke test, then set to `None` for the full run.

In [ ]:
import json
import os
import shutil
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    from huggingface_hub import login

    login(token=userdata.get('HF_TOKEN'))
    drive.mount('/content/drive')

    if os.path.basename(os.getcwd()) != 'stego_CoT':
        if not os.path.exists('stego_CoT'):
            os.system('git clone -q https://github.com/annareshetnyak799-netizen/stego_CoT.git')
        os.chdir('stego_CoT')

    os.system('pip install -q "transformers==5.9.0" accelerate scipy matplotlib')

    DRIVE_DIR = '/content/drive/MyDrive/stego_cot_results/exp14'
    os.makedirs(DRIVE_DIR, exist_ok=True)

    input_local = 'results/exp11/exp11_pairs_with_control.json'
    if not os.path.exists(input_local):
        src = '/content/drive/MyDrive/stego_cot_results/exp11/exp11_pairs_with_control.json'
        os.makedirs('results/exp11', exist_ok=True)
        shutil.copy(src, input_local)
        print('Restored input data from Drive.')
else:
    DRIVE_DIR = None


def backup_to_drive(output_dir, label=''):
    """Copy all exp14* files from output_dir to Drive. No-op outside Colab."""
    if not (IN_COLAB and DRIVE_DIR):
        return
    copied = []
    for fp in Path(output_dir).glob('exp14*'):
        dest = os.path.join(DRIVE_DIR, fp.name)
        shutil.copy(fp, dest)
        copied.append(fp.name)
    tag = f'[{label}] ' if label else ''
    print(f'{tag}Backed up {len(copied)} file(s) to {DRIVE_DIR}')


print('cwd:', os.getcwd())

In [ ]:
from exp14_mechanistic_validation import (
    DEFAULT_OUTPUT_DIR,
    build_cases,
    build_letter_token_tensors,
    load_env,
    load_model_and_tokenizer,
    run_attention_rewrite,
    run_head_specialization,
    run_previous_token_vs_long_range,
    run_scheme_granularity,
    run_source_masking,
    save_json,
)

load_env()

MAX_PAIRS      = 120   # set to None for full 604-pair run; use 20 for smoke test
TOP_K_HEADS    = 8
RANDOM_HEAD_SETS = 8
HEAD_RANK_CASES  = 60  # rank heads on first 60 cases, ablate on the rest
SEED           = 42
OUTPUT_DIR     = DEFAULT_OUTPUT_DIR

tokenizer, model, device = load_model_and_tokenizer()
cases = build_cases(tokenizer=tokenizer, max_pairs=MAX_PAIRS, seed=SEED)
letter_token_tensors = build_letter_token_tensors(tokenizer, device)

print('device:', device)
print('pairs:', len({case.pair_id for case in cases}))
print('cases:', len(cases))

## Notes on the formulas

For one paragraph-start query position `q` and required letter `L`:

- `P(letter | q) = sum softmax(logits[q])[token_id]` over all tokens whose decoded form begins with `L`
- `baseline_mean = average of P(letter | q)` across the paragraph starts in the example, then across examples
- `changed_mean` is the same quantity after an intervention
- `delta = baseline_mean - changed_mean`

Why this metric:

- it directly measures how much the model still supports the **required first letter**
- it is more stable than checking only one exact token string
- it uses the same teacher-forced successful traces as `exp13b`, `exp13c`, and `exp13d`

Why paired statistics:

- each intervention is evaluated on the **same example** as its own baseline
- this removes most task-level variance and isolates the intervention effect

In [ ]:
# exp14a — source masking
# Masks attention to: (1) previous paragraph starts, (2) payload letters only, (3) full scheme span.
# If delta is large -> that source is causally necessary for letter prediction.
summary_a = run_source_masking(
    model=model,
    cases=cases,
    letter_token_tensors=letter_token_tensors,
    output_dir=OUTPUT_DIR,
)
backup_to_drive(OUTPUT_DIR, 'exp14a')
print(json.dumps(summary_a, indent=2)[:3000])

In [ ]:
# exp14b — previous token vs long-range structure
# prev_token_only: masks the last content token before the paragraph break (query_pos - 1)
# prev_starts_only: masks all previous paragraph-start positions
# both: masks both simultaneously
summary_b = run_previous_token_vs_long_range(
    model=model,
    cases=cases,
    letter_token_tensors=letter_token_tensors,
    output_dir=OUTPUT_DIR,
)
backup_to_drive(OUTPUT_DIR, 'exp14b')
print(json.dumps(summary_b, indent=2)[:3000])

In [ ]:
# exp14c — scheme granularity
# Do the specific payload letter tokens carry the constraint, or the surrounding scheme text?
summary_c = run_scheme_granularity(
    model=model,
    cases=cases,
    letter_token_tensors=letter_token_tensors,
    output_dir=OUTPUT_DIR,
)
backup_to_drive(OUTPUT_DIR, 'exp14c')
print(json.dumps(summary_c, indent=2)[:3000])

In [ ]:
# exp14d — head specialization
# Heads ranked on cases[:HEAD_RANK_CASES], ablated on cases[HEAD_RANK_CASES:].
# Compares top-8 heads (by source-attention score) vs 8 random sets of 8 heads.
summary_d = run_head_specialization(
    model=model,
    cases=cases,
    letter_token_tensors=letter_token_tensors,
    output_dir=OUTPUT_DIR,
    top_k=TOP_K_HEADS,
    random_sets=RANDOM_HEAD_SETS,
    head_rank_cases=HEAD_RANK_CASES,
    seed=SEED,
)
backup_to_drive(OUTPUT_DIR, 'exp14d')
print(json.dumps(summary_d, indent=2)[:3000])

In [ ]:
# exp14e — attention rewrite
# Redirects attention mass from source positions uniformly to all non-source positions.
summary_e = run_attention_rewrite(
    model=model,
    cases=cases,
    letter_token_tensors=letter_token_tensors,
    output_dir=OUTPUT_DIR,
)

summary = {
    'max_pairs': MAX_PAIRS,
    'top_k_heads': TOP_K_HEADS,
    'random_head_sets': RANDOM_HEAD_SETS,
    'head_rank_cases': HEAD_RANK_CASES,
    'n_pairs': len({case.pair_id for case in cases}),
    'n_cases': len(cases),
    'experiments': {
        'source_masking': summary_a,
        'prev_token_vs_long_range': summary_b,
        'scheme_granularity': summary_c,
        'head_specialization': summary_d,
        'attention_rewrite': summary_e,
    },
}

save_json(OUTPUT_DIR / 'exp14_summary.json', summary)
backup_to_drive(OUTPUT_DIR, 'exp14e + final')
print('Saved:', OUTPUT_DIR / 'exp14_summary.json')
print(json.dumps(summary, indent=2)[:4000])